In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1yff39f7GbeX88aAjRMXVKxSCVpttD5t2", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))


# Tensor-Parallel Multi-Head Self-Attention

*Part 3 of the Vizuara Tensor Parallelism series*  
*Estimated time: 45-55 minutes*

In this notebook, you will implement **tensor-parallel multi-head self-attention** from scratch. The key insight is that attention heads are already independent, making tensor parallelism exceptionally natural for attention:

1. **Column-parallel Q, K, V projections** -- each GPU gets a subset of attention heads
2. **Local attention computation** -- each GPU runs attention for its heads independently
3. **Row-parallel output projection** -- partial results are combined with a single allreduce

We will then build a **complete tensor-parallel transformer layer** combining attention and MLP, showing that the entire layer requires only **two allreduce operations**.

**Prerequisites:** Notebooks 01 and 02 (column-parallel, row-parallel layers, Megatron MLP block).

In [ ]:
#@title 🎧 Code Walkthrough: Ai Ta And Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_ai_ta_and_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** -- it has already read this entire notebook and can help with concepts, code, and exercises.

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/gpu-programming/tensor-parallelism/practice)**

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
print("Setup complete!")

---


In [ ]:
#@title 🎧 Listen: Attention Review Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_attention_review_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 1: Standard Multi-Head Attention Review

Before parallelizing, let us review multi-head attention.

Given input $X \in \mathbb{R}^{b \times s \times h}$:
1. Project to Q, K, V: $Q = XW_Q$, $K = XW_K$, $V = XW_V$, each $W \in \mathbb{R}^{h \times h}$
2. Split into $A$ heads: each head $i$ has $Q_i, K_i, V_i \in \mathbb{R}^{b \times s \times d_k}$ where $d_k = h/A$
3. Compute attention per head: $\text{head}_i = \text{softmax}(Q_i K_i^T / \sqrt{d_k}) V_i$
4. Concatenate heads and project: $\text{Attn}(X) = \text{concat}(\text{head}_1, ..., \text{head}_A) W_O$

In [ ]:
#@title 🎧 Code Walkthrough: Attention Review Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_attention_review_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class StandardMultiHeadAttention(nn.Module):
    """Standard (non-parallel) multi-head self-attention."""
    
    def __init__(self, hidden_dim: int, num_heads: int):
        super().__init__()
        assert hidden_dim % num_heads == 0
        
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        # QKV projections
        self.W_Q = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, hidden_dim, bias=False)
        
        # Output projection
        self.W_O = nn.Linear(hidden_dim, hidden_dim, bias=False)
    
    def forward(self, x):
        b, s, h = x.shape
        
        # Project to Q, K, V
        Q = self.W_Q(x)  # (b, s, h)
        K = self.W_K(x)
        V = self.W_V(x)
        
        # Reshape for multi-head: (b, s, h) -> (b, num_heads, s, head_dim)
        Q = Q.view(b, s, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(b, s, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(b, s, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)  # (b, num_heads, s, head_dim)
        
        # Concatenate heads: (b, num_heads, s, head_dim) -> (b, s, h)
        attn_output = attn_output.transpose(1, 2).contiguous().view(b, s, h)
        
        # Output projection
        output = self.W_O(attn_output)
        return output


# Create reference attention
h = 64
num_heads = 8
batch, seq = 2, 16

torch.manual_seed(42)
attn_ref = StandardMultiHeadAttention(h, num_heads).to(device)
X = torch.randn(batch, seq, h, device=device)

Y_ref = attn_ref(X)
print(f"Input:  {X.shape}")
print(f"Output: {Y_ref.shape}")
print(f"Heads: {num_heads}, head_dim: {h // num_heads}")
print(f"Parameters: {sum(p.numel() for p in attn_ref.parameters()):,}")

---


In [ ]:
#@title 🎧 Listen: Why Attention Heads Map
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_why_attention_heads_map.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 2: Why Attention Heads Map Naturally to GPUs

The key insight is that in multi-head attention, **each head operates completely independently**. Head $i$ uses only:
- Its slice of Q: columns $[i \cdot d_k : (i+1) \cdot d_k]$ of $W_Q$
- Its slice of K: columns $[i \cdot d_k : (i+1) \cdot d_k]$ of $W_K$
- Its slice of V: columns $[i \cdot d_k : (i+1) \cdot d_k]$ of $W_V$

So the QKV projections are naturally **column-parallel** -- each GPU computes a different subset of heads.

Let us visualize this mapping.

In [ ]:
#@title 🎧 Code Walkthrough: Visualize Head Assignment
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_visualize_head_assignment.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize head-to-GPU assignment
num_heads_viz = 8
h_viz = 64
d_k = h_viz // num_heads_viz

for T in [2, 4, 8]:
    heads_per_gpu = num_heads_viz // T
    print(f"\nT={T} GPUs, {num_heads_viz} heads -> {heads_per_gpu} heads/GPU")
    for gpu in range(T):
        head_start = gpu * heads_per_gpu
        head_end = (gpu + 1) * heads_per_gpu
        col_start = head_start * d_k
        col_end = head_end * d_k
        print(f"  GPU {gpu}: heads [{head_start}:{head_end}], "
              f"W_Q/K/V columns [{col_start}:{col_end}], "
              f"W_O rows [{col_start}:{col_end}]")

---


In [ ]:
#@title 🎧 Listen: Tp Attention Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_tp_attention_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 3: Tensor-Parallel Attention Implementation

Each GPU stores:
- Column shards of $W_Q$, $W_K$, $W_V$ (for its assigned heads)
- Row shard of $W_O$ (corresponding to its assigned heads)

The computation flow on each GPU:
1. Column-parallel QKV: produce Q, K, V for local heads
2. Run attention locally for those heads
3. Row-parallel output projection: produce a partial sum
4. Allreduce: combine partial sums

In [ ]:
#@title 🎧 Code Walkthrough: Tp Attention Class Main
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_tp_attention_class_main.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class TensorParallelAttention(nn.Module):
    """Tensor-parallel multi-head attention for a single GPU shard.
    
    Each GPU handles num_heads // world_size attention heads.
    """
    
    def __init__(self, hidden_dim: int, num_heads: int,
                 world_size: int, rank: int):
        super().__init__()
        assert num_heads % world_size == 0, \
            f"num_heads ({num_heads}) must be divisible by world_size ({world_size})"
        
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.world_size = world_size
        self.rank = rank
        
        self.heads_per_gpu = num_heads // world_size
        self.head_dim = hidden_dim // num_heads
        self.local_hidden = self.heads_per_gpu * self.head_dim  # h / T
        
        # Column-parallel QKV projections
        # Each stores columns for its assigned heads
        self.W_Q = nn.Parameter(torch.empty(hidden_dim, self.local_hidden))
        self.W_K = nn.Parameter(torch.empty(hidden_dim, self.local_hidden))
        self.W_V = nn.Parameter(torch.empty(hidden_dim, self.local_hidden))
        
        # Row-parallel output projection
        # Each stores rows for its assigned heads
        self.W_O = nn.Parameter(torch.empty(self.local_hidden, hidden_dim))
        
        # Initialize
        for p in [self.W_Q, self.W_K, self.W_V, self.W_O]:
            nn.init.kaiming_uniform_(p)
    
    def forward(self, x):
        """Forward pass for this GPU's attention heads.
        
        Args:
            x: Replicated input (batch, seq, hidden_dim)
        Returns:
            Partial output (batch, seq, hidden_dim) -- needs allreduce!
        """
        b, s, h = x.shape
        
        # Step 1: Column-parallel QKV projections
        Q = x @ self.W_Q  # (b, s, local_hidden)
        K = x @ self.W_K
        V = x @ self.W_V
        
        # Step 2: Reshape for local multi-head attention
        Q = Q.view(b, s, self.heads_per_gpu, self.head_dim).transpose(1, 2)
        K = K.view(b, s, self.heads_per_gpu, self.head_dim).transpose(1, 2)
        V = V.view(b, s, self.heads_per_gpu, self.head_dim).transpose(1, 2)
        # Shape: (b, heads_per_gpu, s, head_dim)
        
        # Step 3: Local attention computation (no communication!)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)
        # Shape: (b, heads_per_gpu, s, head_dim)
        
        # Step 4: Concatenate local heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(b, s, self.local_hidden)
        # Shape: (b, s, local_hidden = h/T)
        
        # Step 5: Row-parallel output projection
        output = attn_output @ self.W_O  # (b, s, hidden_dim) -- partial sum!
        
        return output  # Caller must allreduce


In [ ]:
#@title 🎧 Code Walkthrough: Tp Attention Class Helper
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_tp_attention_class_helper.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
    @staticmethod
    def from_standard_attention(attn: StandardMultiHeadAttention,
                                 world_size: int, rank: int):
        """Create from a standard attention module by sharding weights."""
        h = attn.hidden_dim
        A = attn.num_heads
        d_k = h // A
        heads_per_gpu = A // world_size
        local_h = heads_per_gpu * d_k
        
        layer = TensorParallelAttention(h, A, world_size, rank)
        
        col_start = rank * local_h
        col_end = (rank + 1) * local_h
        
        # Column shard of QKV (nn.Linear stores weight as (out, in))
        layer.W_Q = nn.Parameter(attn.W_Q.weight.data[col_start:col_end, :].T.clone())
        layer.W_K = nn.Parameter(attn.W_K.weight.data[col_start:col_end, :].T.clone())
        layer.W_V = nn.Parameter(attn.W_V.weight.data[col_start:col_end, :].T.clone())
        
        # Row shard of output projection
        layer.W_O = nn.Parameter(attn.W_O.weight.data[:, col_start:col_end].T.clone())
        
        return layer


def simulated_allreduce(tensors):
    """Simulate allreduce (sum) across GPUs."""
    return sum(tensors)

print("TensorParallelAttention defined.")

In [ ]:
#@title 🎧 Code Walkthrough: Test Tp Attention
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_test_tp_attention.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Test tensor-parallel attention
T = 2

tp_attn_shards = [
    TensorParallelAttention.from_standard_attention(attn_ref, T, rank).to(device)
    for rank in range(T)
]

print(f"Tensor parallelism degree: {T}")
print(f"Total heads: {num_heads}")
print(f"Heads per GPU: {num_heads // T}")
print(f"\nWeight shapes per GPU:")
for rank in range(T):
    shard = tp_attn_shards[rank]
    print(f"  GPU {rank}:")
    print(f"    W_Q: {shard.W_Q.shape}  (column shard)")
    print(f"    W_K: {shard.W_K.shape}")
    print(f"    W_V: {shard.W_V.shape}")
    print(f"    W_O: {shard.W_O.shape}  (row shard)")

# Forward pass
partial_outputs = [shard(X) for shard in tp_attn_shards]
Y_tp = simulated_allreduce(partial_outputs)

print(f"\nOutput shape: {Y_tp.shape}")
print(f"Matches reference: {torch.allclose(Y_tp, Y_ref, atol=1e-4)}")
print(f"Max difference: {(Y_tp - Y_ref).abs().max().item():.2e}")

The tensor-parallel attention produces the exact same result with only **one allreduce**.


In [ ]:
#@title 🎧 Before You Start: Todo Ex1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_todo_ex1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 1: Attention with T=4

Test tensor-parallel attention with T=4 GPUs. Use a model with 16 heads.

In [ ]:
#@title 🎧 Before You Start: Todo Ex1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_todo_ex1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 1: Tensor-parallel attention with T=4

h_ex1 = 128
heads_ex1 = 16
T_ex1 = 4
b_ex1, s_ex1 = 2, 8

torch.manual_seed(123)
attn_ref_ex1 = StandardMultiHeadAttention(h_ex1, heads_ex1).to(device)
X_ex1 = torch.randn(b_ex1, s_ex1, h_ex1, device=device)
Y_ref_ex1 = attn_ref_ex1(X_ex1)

# ---- YOUR CODE HERE ----
# 1. Create T=4 TensorParallelAttention shards from attn_ref_ex1
# 2. Run forward pass on each shard
# 3. Allreduce partial outputs
# 4. Verify output matches Y_ref_ex1
# 5. Print how many heads each GPU handles



# ---- END YOUR CODE ----

In [ ]:
#@title 🎧 Code Walkthrough: Todo Ex1 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_todo_ex1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Solution for Exercise 1
tp_shards_ex1 = [
    TensorParallelAttention.from_standard_attention(attn_ref_ex1, T_ex1, rank).to(device)
    for rank in range(T_ex1)
]

partials_ex1 = [shard(X_ex1) for shard in tp_shards_ex1]
Y_tp_ex1 = simulated_allreduce(partials_ex1)

print(f"T={T_ex1}, {heads_ex1} total heads, {heads_ex1 // T_ex1} heads per GPU")
print(f"Output shape: {Y_tp_ex1.shape}")
print(f"Matches reference: {torch.allclose(Y_tp_ex1, Y_ref_ex1, atol=1e-4)}")
print(f"\nWeight per GPU: Q={tp_shards_ex1[0].W_Q.shape}, O={tp_shards_ex1[0].W_O.shape}")
ref_params = sum(p.numel() for p in attn_ref_ex1.parameters())
gpu_params = sum(p.numel() for p in tp_shards_ex1[0].parameters())
print(f"Parameters: {ref_params:,} total -> {gpu_params:,} per GPU ({100*gpu_params/ref_params:.0f}%)")

---


In [ ]:
#@title 🎧 Listen: Full Transformer Layer Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_full_transformer_layer_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 4: Complete Tensor-Parallel Transformer Layer

Now let us put it all together: a full transformer layer with tensor-parallel attention AND MLP.

The structure is:
1. LayerNorm (local)
2. Tensor-parallel self-attention -> **AllReduce #1**
3. Residual add (local)
4. LayerNorm (local)
5. Tensor-parallel MLP -> **AllReduce #2**
6. Residual add (local)

**Total communication: exactly 2 allreduces per transformer layer.**

In [ ]:
#@title 🎧 Code Walkthrough: Full Transformer Layer Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_full_transformer_layer_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class TensorParallelTransformerLayer(nn.Module):
    """A single tensor-parallel transformer layer (one GPU shard).
    
    Contains tensor-parallel attention and MLP sub-blocks,
    each with their own LayerNorm and residual connection.
    """
    
    def __init__(self, hidden_dim: int, num_heads: int,
                 world_size: int, rank: int):
        super().__init__()
        self.world_size = world_size
        self.rank = rank
        
        intermediate_dim = 4 * hidden_dim
        
        # Attention sub-block
        self.attn_norm = nn.LayerNorm(hidden_dim)
        self.attention = TensorParallelAttention(
            hidden_dim, num_heads, world_size, rank
        )
        
        # MLP sub-block
        self.mlp_norm = nn.LayerNorm(hidden_dim)
        self.mlp_fc1_weight = nn.Parameter(
            torch.empty(hidden_dim, intermediate_dim // world_size)
        )
        self.mlp_fc2_weight = nn.Parameter(
            torch.empty(intermediate_dim // world_size, hidden_dim)
        )
        
        nn.init.kaiming_uniform_(self.mlp_fc1_weight)
        nn.init.kaiming_uniform_(self.mlp_fc2_weight)
    
    def forward(self, x):
        """Forward pass for one GPU shard.
        
        Returns two partial outputs (attention and MLP) that each need allreduce,
        plus residuals for adding after allreduce.
        """
        # --- Attention sub-block ---
        residual_1 = x
        normed_1 = self.attn_norm(x)  # LayerNorm: local
        attn_partial = self.attention(normed_1)  # Partial sum from attention
        # Caller: x = allreduce(attn_partial) + residual_1
        
        return attn_partial, residual_1
    
    def forward_mlp(self, x):
        """MLP sub-block forward (called after attention allreduce + residual)."""
        residual_2 = x
        normed_2 = self.mlp_norm(x)  # LayerNorm: local
        
        # Column-parallel first layer + GeLU
        z = F.gelu(normed_2 @ self.mlp_fc1_weight)
        
        # Row-parallel second layer
        mlp_partial = z @ self.mlp_fc2_weight  # Partial sum
        # Caller: output = allreduce(mlp_partial) + residual_2
        
        return mlp_partial, residual_2


In [ ]:
#@title 🎧 Code Walkthrough: Full Transformer Layer Run Sim
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_full_transformer_layer_run_sim.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def run_tensor_parallel_layer(shards, x):
    """Simulate running a tensor-parallel transformer layer.
    
    This function simulates what would happen across multiple GPUs.
    """
    T = len(shards)
    
    # Step 1: Attention (column-parallel QKV + local attention + row-parallel O)
    attn_partials = []
    attn_residuals = []
    for shard in shards:
        partial, residual = shard(x)
        attn_partials.append(partial)
        attn_residuals.append(residual)
    
    # AllReduce #1: attention output
    attn_output = simulated_allreduce(attn_partials)
    x = attn_output + attn_residuals[0]  # Residual add (local)
    
    # Step 2: MLP (column-parallel fc1 + GeLU + row-parallel fc2)
    mlp_partials = []
    mlp_residuals = []
    for shard in shards:
        partial, residual = shard.forward_mlp(x)
        mlp_partials.append(partial)
        mlp_residuals.append(residual)
    
    # AllReduce #2: MLP output
    mlp_output = simulated_allreduce(mlp_partials)
    x = mlp_output + mlp_residuals[0]  # Residual add (local)
    
    return x

print("TensorParallelTransformerLayer defined.")

In [ ]:
#@title 🎧 Code Walkthrough: Test Full Layer
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_test_full_layer.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Build and test the full transformer layer
T = 2
h_tf = 64
heads_tf = 8
batch_tf, seq_tf = 2, 16

torch.manual_seed(42)
X_tf = torch.randn(batch_tf, seq_tf, h_tf, device=device)

# Create shards
shards = [
    TensorParallelTransformerLayer(h_tf, heads_tf, T, rank).to(device)
    for rank in range(T)
]

# Synchronize LayerNorm parameters across GPUs
for shard in shards[1:]:
    shard.attn_norm.load_state_dict(shards[0].attn_norm.state_dict())
    shard.mlp_norm.load_state_dict(shards[0].mlp_norm.state_dict())

# Run the layer
output = run_tensor_parallel_layer(shards, X_tf)

print(f"Input:  {X_tf.shape}")
print(f"Output: {output.shape}")
print(f"\nCommunication per layer:")
print(f"  AllReduce #1 (attention): {batch_tf * seq_tf * h_tf * 2 / 1e6:.2f} MB")
print(f"  AllReduce #2 (MLP):       {batch_tf * seq_tf * h_tf * 2 / 1e6:.2f} MB")
print(f"  Total per layer:          {2 * batch_tf * seq_tf * h_tf * 2 / 1e6:.2f} MB")

# Parameter count
total_params = sum(p.numel() for shard in shards for p in shard.parameters())
params_per_gpu = sum(p.numel() for p in shards[0].parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Parameters per GPU: {params_per_gpu:,}")

---


In [ ]:
#@title 🎧 Listen: Comm Analysis Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_comm_analysis_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 5: Communication Analysis for Multi-Layer Models

Let us analyze the communication cost across an entire model.

In [ ]:
#@title 🎧 Code Walkthrough: Comm Analysis Function
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_comm_analysis_function.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def analyze_full_model_communication(model_name, hidden_dim, num_layers, num_heads,
                                      batch, seq_len, tp_degree):
    """Analyze communication cost for a full model with tensor parallelism."""
    
    # Each allreduce: batch * seq * hidden * 2 bytes (FP16)
    ar_size_bytes = batch * seq_len * hidden_dim * 2
    
    # 2 allreduces per layer, forward + backward = 4 per layer
    ar_per_layer = 4  # 2 forward + 2 backward
    total_ar = ar_per_layer * num_layers
    
    # Ring allreduce data volume
    ring_factor = 2 * (tp_degree - 1) / tp_degree
    total_data_bytes = total_ar * ar_size_bytes * ring_factor
    total_data_gb = total_data_bytes / 1e9
    
    # Time estimates
    nvlink_bw = 300e9  # bytes/s
    ib_bw = 25e9       # bytes/s
    
    nvlink_time_ms = total_data_bytes / nvlink_bw * 1000
    ib_time_ms = total_data_bytes / ib_bw * 1000
    
    # FLOPS estimate (very rough)
    # Each layer: ~12 * hidden^2 * batch * seq FLOPs (attention + MLP)
    flops_per_layer = 12 * hidden_dim**2 * batch * seq_len
    total_flops = flops_per_layer * num_layers * 3  # forward + 2x backward
    
    # A100 peak: 312 TFLOPS (FP16), realistic: ~200 TFLOPS
    compute_time_ms = total_flops / (200e12) * 1000
    
    print(f"\n{'='*60}")
    print(f"{model_name} (T={tp_degree}, b={batch}, s={seq_len})")
    print(f"{'='*60}")
    print(f"AllReduces per training step:      {total_ar}")
    print(f"Data per allreduce:                {ar_size_bytes / 1e6:.1f} MB")
    print(f"Total communication volume:        {total_data_gb:.2f} GB")
    print(f"")
    print(f"Estimated times:")
    print(f"  Compute (A100 @ 200 TFLOPS):     {compute_time_ms:.1f} ms")
    print(f"  Communication (NVLink 300 GB/s):  {nvlink_time_ms:.1f} ms")
    print(f"  Communication (IB 25 GB/s):       {ib_time_ms:.1f} ms")
    print(f"")
    print(f"  Comm/Compute ratio (NVLink):      {nvlink_time_ms/compute_time_ms:.2%}")
    print(f"  Comm/Compute ratio (IB):          {ib_time_ms/compute_time_ms:.2%}")

# Analyze different configurations
configs = [
    ('LLaMA 7B',   4096, 32, 32, 4, 2048, 4),
    ('LLaMA 7B',   4096, 32, 32, 4, 2048, 8),
    ('LLaMA 70B',  8192, 80, 64, 1, 4096, 8),
    ('GPT-3 175B', 12288, 96, 96, 1, 2048, 8),
]

for args in configs:
    analyze_full_model_communication(*args)

In [ ]:
#@title 🎧 Listen: Comm Analysis Results
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_comm_analysis_results.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


The numbers show clearly why tensor parallelism is confined to within a single node:
- On NVLink: communication overhead is a small fraction of compute time
- On InfiniBand: communication overhead can be comparable to or exceed compute time

---


In [ ]:
#@title 🎧 Transition: Stacking Layers Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_stacking_layers_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 6: Stacking Multiple Layers

Let us build a multi-layer model and trace the allreduces.

In [ ]:
#@title 🎧 Code Walkthrough: Stacking Layers Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_stacking_layers_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class TensorParallelModel(nn.Module):
    """A multi-layer tensor-parallel transformer model (one GPU shard)."""
    
    def __init__(self, hidden_dim: int, num_heads: int, num_layers: int,
                 world_size: int, rank: int):
        super().__init__()
        self.layers = nn.ModuleList([
            TensorParallelTransformerLayer(hidden_dim, num_heads, world_size, rank)
            for _ in range(num_layers)
        ])
        self.world_size = world_size
        self.rank = rank
    
    def forward(self, x):
        """Forward pass through all layers. Returns list of partial outputs."""
        all_attn_partials = []
        all_mlp_partials = []
        
        for layer in self.layers:
            attn_partial, _ = layer(x)
            all_attn_partials.append(attn_partial)
            # In real code, allreduce happens here
            # For simulation, we just return partials
        
        return all_attn_partials, all_mlp_partials


# Build a small multi-layer model
T = 2
h_ml = 64
heads_ml = 8
num_layers = 4
batch_ml, seq_ml = 2, 16

torch.manual_seed(42)

# Create model shards
model_shards = [
    TensorParallelModel(h_ml, heads_ml, num_layers, T, rank).to(device)
    for rank in range(T)
]

# Sync LayerNorm params
for rank in range(1, T):
    for l in range(num_layers):
        model_shards[rank].layers[l].attn_norm.load_state_dict(
            model_shards[0].layers[l].attn_norm.state_dict()
        )
        model_shards[rank].layers[l].mlp_norm.load_state_dict(
            model_shards[0].layers[l].mlp_norm.state_dict()
        )

# Run through all layers with proper allreduces
X_ml = torch.randn(batch_ml, seq_ml, h_ml, device=device)
x = X_ml
allreduce_count = 0

for layer_idx in range(num_layers):
    # Attention
    attn_partials = [model_shards[r].layers[layer_idx](x)[0] for r in range(T)]
    attn_residual = x
    x = simulated_allreduce(attn_partials) + attn_residual
    allreduce_count += 1
    
    # MLP
    mlp_partials = [model_shards[r].layers[layer_idx].forward_mlp(x)[0] for r in range(T)]
    mlp_residual = x
    x = simulated_allreduce(mlp_partials) + mlp_residual
    allreduce_count += 1

print(f"Model: {num_layers} layers, h={h_ml}, {heads_ml} heads, T={T}")
print(f"Output shape: {x.shape}")
print(f"Total allreduces in forward pass: {allreduce_count}")
print(f"Expected: 2 * {num_layers} = {2 * num_layers}")

# Parameters
params_total = sum(p.numel() for shard in model_shards for p in shard.parameters()) // T
params_per_gpu = sum(p.numel() for p in model_shards[0].parameters())
print(f"\nUnique parameters: ~{params_total:,}")
print(f"Parameters per GPU: {params_per_gpu:,}")

---


In [ ]:
#@title 🎧 Transition: Profiling Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_profiling_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 7: Profiling the Full Layer

Let us measure forward pass time for a single transformer layer at different sizes.

In [ ]:
#@title 🎧 Code Walkthrough: Profiling Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_profiling_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def profile_transformer_layer(hidden_dim, num_heads, batch, seq_len, tp_degree, num_iters=30):
    """Profile a single tensor-parallel transformer layer."""
    torch.manual_seed(42)
    
    shards = [
        TensorParallelTransformerLayer(hidden_dim, num_heads, tp_degree, rank).to(device)
        for rank in range(tp_degree)
    ]
    
    X_prof = torch.randn(batch, seq_len, hidden_dim, device=device)
    
    # Warmup
    for _ in range(3):
        _ = run_tensor_parallel_layer(shards, X_prof)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(num_iters):
        _ = run_tensor_parallel_layer(shards, X_prof)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    elapsed_ms = (time.perf_counter() - start) / num_iters * 1000
    
    params = sum(p.numel() for p in shards[0].parameters())
    ar_size = batch * seq_len * hidden_dim * 2  # FP16 bytes
    
    return elapsed_ms, params, ar_size


configs = [
    (64,  8,  4, 32, 2),
    (128, 8,  4, 32, 2),
    (256, 16, 4, 64, 4),
    (512, 16, 2, 64, 4),
    (1024, 16, 2, 128, 8),
]

print(f"{'h':>6} {'heads':>6} {'b':>4} {'s':>5} {'T':>3} {'Time/Layer':>12} {'Params/GPU':>12} {'AR Size':>10}")
print("-" * 70)
for h, heads, b, s, T in configs:
    t, params, ar = profile_transformer_layer(h, heads, b, s, T)
    print(f"{h:>6} {heads:>6} {b:>4} {s:>5} {T:>3} {t:>10.3f}ms {params:>12,} {ar/1e6:>8.2f}MB")

In [ ]:
#@title 🎧 Before You Start: Todo Ex2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_todo_ex2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 2: Implement and Test the Complete Flow

Build a complete tensor-parallel transformer layer from scratch (not using the pre-built classes), run forward and backward passes, and verify correctness.

In [ ]:
#@title 🎧 Before You Start: Todo Ex2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_todo_ex2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 2: Manual tensor-parallel forward pass
# Implement the full tensor-parallel transformer layer step by step,
# without using the TensorParallelTransformerLayer class.

h_ex2 = 32
heads_ex2 = 4
T_ex2 = 2
b_ex2, s_ex2 = 2, 8

torch.manual_seed(42)
X_ex2 = torch.randn(b_ex2, s_ex2, h_ex2, device=device)

# Create weight matrices for a single transformer layer
# Attention weights
W_Q = torch.randn(h_ex2, h_ex2, device=device)
W_K = torch.randn(h_ex2, h_ex2, device=device)
W_V = torch.randn(h_ex2, h_ex2, device=device)
W_O = torch.randn(h_ex2, h_ex2, device=device)

# MLP weights
W1 = torch.randn(h_ex2, 4 * h_ex2, device=device)
W2 = torch.randn(4 * h_ex2, h_ex2, device=device)

# ---- YOUR CODE HERE ----
# 1. Compute reference output (standard attention + MLP with residuals)
# 2. Split weights for T=2 GPUs
# 3. Run tensor-parallel forward:
#    a. Column-parallel QKV projections
#    b. Local attention per GPU
#    c. Row-parallel output projection
#    d. Allreduce #1 + residual
#    e. Column-parallel MLP fc1 + GeLU
#    f. Row-parallel MLP fc2
#    g. Allreduce #2 + residual
# 4. Verify outputs match



# ---- END YOUR CODE ----

In [ ]:
#@title 🎧 Code Walkthrough: Todo Ex2 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_26_todo_ex2_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Solution for Exercise 2

d_k = h_ex2 // heads_ex2
heads_per_gpu = heads_ex2 // T_ex2
local_h = heads_per_gpu * d_k

# --- Reference computation ---
# Attention
Q_ref = X_ex2 @ W_Q
K_ref = X_ex2 @ W_K
V_ref = X_ex2 @ W_V

Q_ref = Q_ref.view(b_ex2, s_ex2, heads_ex2, d_k).transpose(1, 2)
K_ref = K_ref.view(b_ex2, s_ex2, heads_ex2, d_k).transpose(1, 2)
V_ref = V_ref.view(b_ex2, s_ex2, heads_ex2, d_k).transpose(1, 2)

scores_ref = torch.matmul(Q_ref, K_ref.transpose(-2, -1)) / math.sqrt(d_k)
attn_ref = torch.matmul(F.softmax(scores_ref, dim=-1), V_ref)
attn_ref = attn_ref.transpose(1, 2).contiguous().view(b_ex2, s_ex2, h_ex2)
attn_out_ref = attn_ref @ W_O

x_after_attn_ref = X_ex2 + attn_out_ref  # Residual

# MLP
mlp_ref = F.gelu(x_after_attn_ref @ W1) @ W2
Y_ref_ex2 = x_after_attn_ref + mlp_ref  # Residual

# --- Tensor-parallel computation ---
attn_partials = []
for rank in range(T_ex2):
    col_start = rank * local_h
    col_end = (rank + 1) * local_h
    
    # Column-parallel QKV
    Q_local = X_ex2 @ W_Q[:, col_start:col_end]  # (b, s, local_h)
    K_local = X_ex2 @ W_K[:, col_start:col_end]
    V_local = X_ex2 @ W_V[:, col_start:col_end]
    
    # Reshape for local attention
    Q_local = Q_local.view(b_ex2, s_ex2, heads_per_gpu, d_k).transpose(1, 2)
    K_local = K_local.view(b_ex2, s_ex2, heads_per_gpu, d_k).transpose(1, 2)
    V_local = V_local.view(b_ex2, s_ex2, heads_per_gpu, d_k).transpose(1, 2)
    
    # Local attention
    scores_local = torch.matmul(Q_local, K_local.transpose(-2, -1)) / math.sqrt(d_k)
    attn_local = torch.matmul(F.softmax(scores_local, dim=-1), V_local)
    attn_local = attn_local.transpose(1, 2).contiguous().view(b_ex2, s_ex2, local_h)
    
    # Row-parallel output projection
    partial = attn_local @ W_O[col_start:col_end, :]
    attn_partials.append(partial)

# AllReduce #1 + residual
x_after_attn_tp = simulated_allreduce(attn_partials) + X_ex2

# MLP
inter_per_gpu = (4 * h_ex2) // T_ex2
mlp_partials = []
for rank in range(T_ex2):
    col_start = rank * inter_per_gpu
    col_end = (rank + 1) * inter_per_gpu
    
    # Column-parallel fc1 + GeLU
    z_local = F.gelu(x_after_attn_tp @ W1[:, col_start:col_end])
    
    # Row-parallel fc2
    partial = z_local @ W2[col_start:col_end, :]
    mlp_partials.append(partial)

# AllReduce #2 + residual
Y_tp_ex2 = simulated_allreduce(mlp_partials) + x_after_attn_tp

print(f"Reference output shape: {Y_ref_ex2.shape}")
print(f"TP output shape:        {Y_tp_ex2.shape}")
print(f"Outputs match:          {torch.allclose(Y_ref_ex2, Y_tp_ex2, atol=1e-4)}")
print(f"Max difference:         {(Y_ref_ex2 - Y_tp_ex2).abs().max().item():.2e}")
print(f"\nTotal allreduces used: 2 (one for attention, one for MLP)")

---


In [ ]:
#@title 🎧 Listen: What Cannot Be Parallelized Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_27_what_cannot_be_parallelized_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 8: What Cannot Be Parallelized with Tensor Parallelism

Not every operation is tensor-parallelizable. Let us identify the operations that must run on replicated data.

In [ ]:
#@title 🎧 Code Walkthrough: What Cannot Be Parallelized Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_28_what_cannot_be_parallelized_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Operations that must run on replicated data

torch.manual_seed(42)
x_demo = torch.randn(2, 8, 32, device=device)

# 1. LayerNorm -- requires full feature dimension
ln = nn.LayerNorm(32).to(device)
ln_full = ln(x_demo)

# Try splitting and applying LayerNorm
x_split = torch.chunk(x_demo, 2, dim=-1)
ln_half = nn.LayerNorm(16).to(device)
ln_parts = [ln_half(part) for part in x_split]
ln_split_then_combine = torch.cat(ln_parts, dim=-1)

print("LayerNorm on split data vs full data:")
print(f"  Match: {torch.allclose(ln_full, ln_split_then_combine)}")
print(f"  Reason: LayerNorm normalizes over the FULL feature dimension.")
print(f"  Solution: LayerNorm runs on replicated data (before column-parallel split).")

# 2. Dropout -- must use same random seed across GPUs for replicated data
print(f"\nDropout:")
print(f"  Dropout on replicated data must use synchronized random seeds.")
print(f"  Dropout on split activations (after column-parallel) can use independent seeds.")

# 3. Residual connections -- add replicated data
print(f"\nResidual connections:")
print(f"  x + attn_output: both are replicated after allreduce. Local operation.")

# 4. Embedding layers
print(f"\nEmbeddings:")
print(f"  Can be parallelized along vocabulary dimension (column-parallel).")
print(f"  Or replicated if vocab is small enough.")

---


In [ ]:
#@title 🎧 Listen: Hidden Memory Cost Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_29_hidden_memory_cost_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 9: The Hidden Memory Cost -- Replicated Activations

While tensor parallelism splits the weight matrices, there is a hidden memory cost: certain activations must be **replicated** on every GPU.

In [ ]:
#@title 🎧 Code Walkthrough: Hidden Memory Cost Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_30_hidden_memory_cost_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def analyze_activation_memory(hidden_dim, num_heads, num_layers,
                               batch, seq_len, tp_degree):
    """Analyze activation memory with tensor parallelism."""
    
    inter = 4 * hidden_dim
    d_k = hidden_dim // num_heads
    
    # Per-layer activations (FP16, 2 bytes per element)
    # Replicated activations (same on every GPU):
    ln_input = batch * seq_len * hidden_dim * 2  # LayerNorm input
    residual = batch * seq_len * hidden_dim * 2  # Residual tensor
    replicated_per_layer = 2 * (ln_input + residual)  # 2 sub-blocks
    
    # Split activations (1/T on each GPU):
    qkv = 3 * batch * seq_len * (hidden_dim // tp_degree) * 2
    attn_scores = batch * (num_heads // tp_degree) * seq_len * seq_len * 2
    attn_output = batch * seq_len * (hidden_dim // tp_degree) * 2
    mlp_inter = batch * seq_len * (inter // tp_degree) * 2
    split_per_layer = qkv + attn_scores + attn_output + mlp_inter
    
    total_per_layer = replicated_per_layer + split_per_layer
    total_all_layers = total_per_layer * num_layers
    
    print(f"\nActivation memory per layer (T={tp_degree}):")
    print(f"  Replicated (same on all GPUs): {replicated_per_layer / 1e6:.1f} MB")
    print(f"  Split (1/T per GPU):           {split_per_layer / 1e6:.1f} MB")
    print(f"  Total per GPU per layer:       {total_per_layer / 1e6:.1f} MB")
    print(f"  Total per GPU ({num_layers} layers):    {total_all_layers / 1e9:.2f} GB")
    print(f"")
    print(f"  Replicated fraction:           {replicated_per_layer / total_per_layer:.1%}")
    print(f"  This wasted memory motivates SEQUENCE PARALLELISM (next pod)!")

# LLaMA 70B analysis
print("=" * 60)
print("LLaMA 70B Activation Memory Analysis")
print("=" * 60)
analyze_activation_memory(
    hidden_dim=8192, num_heads=64, num_layers=80,
    batch=1, seq_len=4096, tp_degree=8
)

print("\n" + "=" * 60)
print("GPT-3 175B Activation Memory Analysis")
print("=" * 60)
analyze_activation_memory(
    hidden_dim=12288, num_heads=96, num_layers=96,
    batch=1, seq_len=2048, tp_degree=8
)

The replicated activations (LayerNorm inputs, residual tensors) account for a significant fraction of memory. This is the motivation for **sequence parallelism**, which splits these replicated operations along the sequence dimension -- the topic of the next pod.

---


In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_31_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Summary

In this notebook, you built the complete tensor-parallel transformer layer:

| Component | Strategy | Communication |
|-----------|----------|---------------|
| LayerNorm | Local (replicated) | None |
| QKV projection | Column-parallel | None |
| Attention heads | Local (independent heads) | None |
| Output projection | Row-parallel | **AllReduce #1** |
| Residual add | Local (replicated) | None |
| LayerNorm | Local (replicated) | None |
| MLP fc1 | Column-parallel | None |
| GeLU | Local (element-wise) | None |
| MLP fc2 | Row-parallel | **AllReduce #2** |
| Residual add | Local (replicated) | None |

**Total: 2 allreduces per transformer layer.** This is optimal for tensor parallelism.

**Key takeaways:**
1. Attention heads map naturally to GPUs -- each GPU handles its assigned heads independently
2. The column-parallel -> row-parallel pattern minimizes communication to one allreduce per sub-block
3. Tensor parallelism belongs **within a single node** where NVLink provides high bandwidth
4. The hidden cost is replicated activations, which motivates sequence parallelism